# 6. Run PyNNLF: SA BESS 44hh Clean Cohort

Prepared for manual execution. Runs `ds22`-`ds24` for 1-day-ahead forecasting using the 12-model set.

## 1. Setup And Batch Spec

In [1]:
from pathlib import Path
import sys

def find_publication_project(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "specs").exists() and (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pynnlf").exists():
            return candidate
    raise FileNotFoundError("Could not find the PyNNLF repo root.")
PROJECT_DIR = find_publication_project()
REPO_ROOT = find_repo_root(PROJECT_DIR)
print(f"Publication project: {PROJECT_DIR}")
print(f"Repository root: {REPO_ROOT}")
import pandas as pd
import yaml
sys.path.insert(0, str(REPO_ROOT / "src"))
import pynnlf  # noqa: E402
BATCH_PATH = PROJECT_DIR / "specs" / "sa_bess_44hh_batch.yaml"
RESULTS_ROOT = PROJECT_DIR / "experiment_result"
TEMP_SPEC_PATH = PROJECT_DIR / "specs" / "_tmp_sa_bess_44hh_single.yaml"
batch = yaml.safe_load(BATCH_PATH.read_text(encoding="utf-8"))
config = yaml.safe_load((PROJECT_DIR / "specs" / "pynnlf_config.yaml").read_text(encoding="utf-8"))
forecast_horizon_minutes = {key: int(value) for key, value in config["forecast_horizons"].items()}
print(batch)

Publication project: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1
Repository root: C:\Users\z5404477\Documents\PyNNLF
{'datasets': ['ds22', 'ds23', 'ds24'], 'forecast_horizons': ['fh8'], 'model_and_hp': [['m1', 'hp1'], ['m2', 'hp2'], ['m3', 'hp1'], ['m4', 'hp1'], ['m6', 'hp1'], ['m7', 'hp1'], ['m8', 'hp1'], ['m9', 'hp3'], ['m10', 'hp1'], ['m13', 'hp2'], ['m16', 'hp1'], ['m17', 'hp1']]}


## 2. Validate Input Datasets

In [2]:
EXPECTED_FILES = {"ds22": "ds22_sa_bess_44hh_pos_underlying_load_30min.csv", "ds23": "ds23_sa_bess_44hh_pos_net_load_with_pv_30min.csv", "ds24": "ds24_sa_bess_44hh_pos_net_load_with_pv_battery_30min.csv"}
for dataset_id, filename in EXPECTED_FILES.items():
    path = PROJECT_DIR / "data" / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing {dataset_id}: {path}. Run notebook 5 first.")
    df = pd.read_csv(path, parse_dates=["datetime"])
    if list(df.columns) != ["datetime", "netload_kW"] or len(df) != 17_520 or df.isna().any().any() or df["datetime"].duplicated().any():
        raise ValueError(f"{filename}: failed basic validation")
    print(f"{dataset_id}: OK | {filename}")

ds22: OK | ds22_sa_bess_44hh_pos_underlying_load_30min.csv
ds23: OK | ds23_sa_bess_44hh_pos_net_load_with_pv_30min.csv
ds24: OK | ds24_sa_bess_44hh_pos_net_load_with_pv_battery_30min.csv


## 3. Run Missing Experiments

This cell runs PyNNLF. It is resumable and skips completed combinations.

In [3]:
def completed_keys(results_root):
    keys = set()
    for result_file in sorted(results_root.glob("E*/E*_a1_experiment_result.csv")):
        try:
            row = pd.read_csv(result_file, nrows=1).iloc[0]
            keys.add((str(row.get("dataset_no", "")), int(row.get("forecast_horizon_min")), str(row.get("model_no", "")), str(row.get("hyperparameter_no", ""))))
        except Exception:
            continue
    return keys
done = completed_keys(RESULTS_ROOT)
total = len(batch["datasets"]) * len(batch["forecast_horizons"]) * len(batch["model_and_hp"])
run_index = 0
for dataset_id in batch["datasets"]:
    for forecast_horizon_id in batch["forecast_horizons"]:
        horizon_minutes = forecast_horizon_minutes[forecast_horizon_id]
        for model_id, hp in batch["model_and_hp"]:
            run_index += 1
            key = (str(dataset_id), horizon_minutes, str(model_id), str(hp))
            if key in done:
                print(f"[skip {run_index}/{total}] {dataset_id} {forecast_horizon_id} {model_id} {hp}")
                continue
            print(f"[run {run_index}/{total}] {dataset_id} {forecast_horizon_id} {model_id} {hp}")
            temp_spec = {"datasets": [dataset_id], "forecast_horizons": [forecast_horizon_id], "model_and_hp": [[model_id, hp]]}
            TEMP_SPEC_PATH.write_text(yaml.safe_dump(temp_spec, sort_keys=False), encoding="utf-8")
            pynnlf.run_experiment_batch(TEMP_SPEC_PATH, plot_enabled=False)
            done.add(key)
if TEMP_SPEC_PATH.exists():
    TEMP_SPEC_PATH.unlink()
pynnlf.recap_experiments(RESULTS_ROOT)
print(f"Recap written: {RESULTS_ROOT / 'a1_experiment_result.csv'}")

[run 1/36] ds22 fh8 m1 hp1
Pass Test 1 - Forecast horizon is >= dataset frequency
Pass Test 2 - Hyperparameter choice is possible given the forecast horizon
Processing CV 1 / 10....

Processing CV 2 / 10....

Processing CV 3 / 10....

Processing CV 4 / 10....

Processing CV 5 / 10....

Processing CV 6 / 10....

Processing CV 7 / 10....

Processing CV 8 / 10....

Processing CV 9 / 10....

Processing CV 10 / 10....

[run 2/36] ds22 fh8 m2 hp2
Pass Test 1 - Forecast horizon is >= dataset frequency
Pass Test 2 - Hyperparameter choice is possible given the forecast horizon
Processing CV 1 / 10....

Processing CV 2 / 10....

Processing CV 3 / 10....

Processing CV 4 / 10....

Processing CV 5 / 10....

Processing CV 6 / 10....

Processing CV 7 / 10....

Processing CV 8 / 10....

Processing CV 9 / 10....

Processing CV 10 / 10....

[run 3/36] ds22 fh8 m3 hp1
Pass Test 1 - Forecast horizon is >= dataset frequency
Pass Test 2 - Hyperparameter choice is possible given the forecast horizon
Process

c:\Users\z5404477\Documents\PyNNLF\.venv\Lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)



Processing CV 2 / 10....


c:\Users\z5404477\Documents\PyNNLF\.venv\Lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)



Processing CV 3 / 10....


...

c:\Users\z5404477\Documents\PyNNLF\.venv\Lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)



Processing CV 10 / 10....


c:\Users\z5404477\Documents\PyNNLF\.venv\Lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)



[run 10/36] ds22 fh8 m13 hp2
Pass Test 1 - Forecast horizon is >= dataset frequency
Pass Test 2 - Hyperparameter choice is possible given the forecast horizon
Processing CV 1 / 10....
Epoch [1/100]
Epoch [1/100] and batch [1/3]
Epoch [1/100] and batch [2/3]
Epoch [1/100] and batch [3/3]
Epoch [1/100], Loss: 2766498.5000, time taken: 33.78 seconds
Epoch [2/100]
Epoch [2/100] and batch [1/3]
Epoch [2/100] and batch [2/3]
Epoch [2/100] and batch [3/3]
Epoch [2/100], Loss: 2551577.2500, time taken: 29.41 seconds
Epoch [3/100]
Epoch [3/100] and batch [1/3]
Epoch [3/100] and batch [2/3]
Epoch [3/100] and batch [3/3]
Epoch [3/100], Loss: 2346384.5000, time taken: 29.12 seconds
Epoch [4/100]
Epoch [4/100] and batch [1/3]
Epoch [4/100] and batch [2/3]
Epoch [4/100] and batch [3/3]
Epoch [4/100], Loss: 2152069.5000, time taken: 28.78 seconds
Epoch [5/100]
Epoch [5/100] and batch [1/3]
Epoch [5/100] and batch [2/3]
Epoch [5/100] and batch [3/3]
Epoch [5/100], Loss: 1967436.3750, time taken: 29.0

KeyboardInterrupt: 